# Free Cursor QLoRA Training (Colab T4)\n\nThis notebook runs the full pipeline end-to-end:\n1. Verify GPU\n2. Install dependencies\n3. Mount Google Drive\n4. Extract dataset archive\n5. Train QLoRA\n6. Evaluate\n7. Merge LoRA and export ONNX bundle\n\nExpected dataset archive in Drive (fallback supported):\n- `/content/drive/MyDrive/free_cursor_dataset.tar.gz`\n- `/content/drive/MyDrive/dataset.tar.gz`

In [ ]:
import os\n\nREPO_URL = "https://github.com/dewd5252/free-cursor-mvp.git"\nREPO_DIR = "/content/free-cursor-mvp"\n\nif not os.path.exists(REPO_DIR):\n    !git clone {REPO_URL} {REPO_DIR}\n\n%cd /content/free-cursor-mvp\nprint("Repo ready at /content/free-cursor-mvp")

In [ ]:
!nvidia-smi\n\nimport torch\nprint("CUDA available:", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets sentencepiece optimum[onnxruntime] onnx onnxruntime

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
import os\nimport tarfile\n\narchive_candidates = [\n    '/content/drive/MyDrive/free_cursor_dataset.tar.gz',\n    '/content/drive/MyDrive/dataset.tar.gz',\n]\n\narchive_path = next((p for p in archive_candidates if os.path.exists(p)), None)\nif archive_path is None:\n    raise FileNotFoundError(\n        'Dataset archive not found. Upload free_cursor_dataset.tar.gz to MyDrive.'\n    )\n\nprint('Using archive:', archive_path)\n\nwith tarfile.open(archive_path, 'r:gz') as tar:\n    tar.extractall('/content')\n\nfor required in [\n    '/content/dataset/splits/train.jsonl',\n    '/content/dataset/splits/val.jsonl',\n    '/content/dataset/splits/test.jsonl',\n]:\n    if not os.path.exists(required):\n        raise FileNotFoundError(f'Missing expected file: {required}')\n\nprint('Dataset extracted successfully.')

In [ ]:
from datasets import load_dataset\n\nds = load_dataset(\n    'json',\n    data_files={\n        'train': '/content/dataset/splits/train.jsonl',\n        'val': '/content/dataset/splits/val.jsonl',\n        'test': '/content/dataset/splits/test.jsonl',\n    },\n)\nprint(ds)\nprint(ds['train'][0])

In [ ]:
!python scripts/ml/colab_train_qlora.py \\n  --model-id Qwen/Qwen2.5-0.5B-Instruct \\n  --train /content/dataset/splits/train.jsonl \\n  --val /content/dataset/splits/val.jsonl \\n  --test /content/dataset/splits/test.jsonl \\n  --max-seq-length 1024 \\n  --epochs 1 \\n  --per-device-batch 4 \\n  --grad-accum 4 \\n  --lr 2e-4 \\n  --lora-output /content/drive/MyDrive/free_cursor_lora_weights \\n  --report-output /content/drive/MyDrive/free_cursor_eval_report.json

In [ ]:
import json\n\nreport_path = '/content/drive/MyDrive/free_cursor_eval_report.json'\nwith open(report_path, 'r', encoding='utf-8') as f:\n    report = json.load(f)\n\nprint(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
!python scripts/ml/merge_and_export_onnx.py \\n  --base-model Qwen/Qwen2.5-0.5B-Instruct \\n  --lora-dir /content/drive/MyDrive/free_cursor_lora_weights \\n  --merged-dir /content/free_cursor_merged \\n  --onnx-dir /content/drive/MyDrive/free_cursor_onnx_bundle

In [ ]:
import os\n\nbundle_dir = '/content/drive/MyDrive/free_cursor_onnx_bundle'\nrequired = [\n    'model.onnx',\n    'tokenizer.json',\n    'tokenizer_config.json',\n    'special_tokens_map.json',\n    'generation_config.json',\n]\n\nmissing = [name for name in required if not os.path.exists(os.path.join(bundle_dir, name))]\nif missing:\n    print('Missing files:', missing)\nelse:\n    print('ONNX bundle looks complete.')\n    for name in required:\n        path = os.path.join(bundle_dir, name)\n        print(f'- {name}: {os.path.getsize(path)} bytes')